# Resume the physical-parameterization UIEB ablation

Attach one stopped notebook output as a Kaggle input, enable Internet and a GPU, set `SEEDS_TO_RUN` and `RESUME_CHECKPOINT` below, and run all cells. Use a separate resume job for each seed. The checkpoint should normally be the interrupted run's `last_model.pth`; outputs from the older notebook may only have `best_model.pth`.

The runner copies the complete previous output tree into `/kaggle/working`, skips completed runs, resumes the interrupted run, and then trains any remaining configurations.

In [ ]:
from pathlib import Path

# Resume only the seed belonging to this attached output (for example [1] or [2]).
SEEDS_TO_RUN = [1]

# Replace this with the newest .pth from that seed's interrupted run.
# Keep the original <output-root>/<configuration>/seed_<n>/... directory layout.
RESUME_CHECKPOINT = Path(
    '/kaggle/input/REPLACE_WITH_PREVIOUS_OUTPUT/physics_parameterization_ablation_uieb/'
    'physical_no_reconstruction/seed_0/best_model.pth'
)
assert len(SEEDS_TO_RUN) == 1, 'Resume one seed per Kaggle job'
assert RESUME_CHECKPOINT.is_file(), f'Edit RESUME_CHECKPOINT; file not found: {RESUME_CHECKPOINT}'

In [ ]:
import os
import subprocess

REPO = Path('/kaggle/working/underwater-image-enhancement')
BRANCH = 'learnable-physics-extractor'
if not REPO.exists():
    subprocess.run([
        'git', 'clone', '--branch', BRANCH, '--single-branch', '--depth', '1',
        'https://github.com/heniath/underwater-image-enhancement.git', str(REPO),
    ], check=True)
os.environ['UWIR_RESUME_CHECKPOINT'] = str(RESUME_CHECKPOINT)
os.environ['UWIR_SEEDS_TO_RUN'] = ','.join(map(str, SEEDS_TO_RUN))
os.chdir(REPO)
runner = REPO / 'learnable_physics_uieb_kaggle.ipynb'
assert runner.is_file(), f'Runner notebook not found on {BRANCH}: {runner}'
get_ipython().run_line_magic('run', f'-i {runner}')